In [1]:
from collections import namedtuple, defaultdict   # added defaultdict
import numpy as np

IGNORE_ID = 255

# ── Extended label namedtuple ─────────────────────────────────────────────────
LabelUnified = namedtuple('LabelUnified', [
    'name', 'id', 'csId', 'csTrainId',
    'level4Id', 'level3Id', 'level2IdName', 'level2Id', 'level1Id',
    'hasInstances', 'ignoreInEval', 'color',
    'unifiedId',
])

labels_unified = [
    #        name                     id   csId  csTrId  l4   l3   l2Name               l2   l1  hasInst ignEval  color               unifiedId
    LabelUnified('road',                  0,   7,   0,   0,   0,  'drivable',           0,   0,  False, False, (128, 64,128),   0  ),
    LabelUnified('parking',               1,   9, 255,   1,   1,  'drivable',           1,   0,  False, False, (250,170,160),   19 ),
    LabelUnified('drivable fallback',     2, 255, 255,   2,   1,  'drivable',           1,   0,  False, False, ( 81,  0, 81),   32 ),
    LabelUnified('sidewalk',              3,   8,   1,   3,   2,  'non-drivable',       2,   1,  False, False, (244, 35,232),   1  ),
    LabelUnified('rail track',            4,  10, 255,   3,   3,  'non-drivable',       3,   1,  False, False, (230,150,140),   20 ),
    LabelUnified('non-drivable fallback', 5, 255,   9,   4,   3,  'non-drivable',       3,   1,  False, False, (152,251,152),   33 ),
    LabelUnified('person',                6,  24,  11,   5,   4,  'living-thing',       4,   2,  True,  False, (220, 20, 60),   11 ),
    LabelUnified('animal',                7, 255, 255,   6,   4,  'living-thing',       4,   2,  True,  True,  (246,198,145),   21 ),
    LabelUnified('rider',                 8,  25,  12,   7,   5,  'living-thing',       5,   2,  True,  False, (255,  0,  0),   12 ),
    LabelUnified('motorcycle',            9,  32,  17,   8,   6,  '2-wheeler',          6,   3,  True,  False, (  0,  0,230),   17 ),
    LabelUnified('bicycle',              10,  33,  18,   9,   7,  '2-wheeler',          6,   3,  True,  False, (119, 11, 32),   18 ),
    LabelUnified('autorickshaw',         11, 255, 255,  10,   8,  'autorickshaw',       7,   3,  True,  False, (255,204, 54),   22 ),
    LabelUnified('car',                  12,  26,  13,  11,   9,  'car',                7,   3,  True,  False, (  0,  0,142),   13 ),
    LabelUnified('truck',                13,  27,  14,  12,  10,  'large-vehicle',      8,   3,  True,  False, (  0,  0, 70),   14 ),
    LabelUnified('bus',                  14,  28,  15,  13,  11,  'large-vehicle',      8,   3,  True,  False, (  0, 60,100),   15 ),
    LabelUnified('caravan',              15,  29, 255,  14,  12,  'large-vehicle',      8,   3,  True,  True,  (  0,  0, 90),   23 ),
    LabelUnified('trailer',              16,  30, 255,  15,  12,  'large-vehicle',      8,   3,  True,  True,  (  0,  0,110),   24 ),
    LabelUnified('train',                17,  31,  16,  15,  12,  'large-vehicle',      8,   3,  True,  True,  (  0, 80,100),   16 ),
    LabelUnified('vehicle fallback',     18, 255, 255,  15,  12,  'large-vehicle',      8,   3,  True,  False, (136,143,153),   30 ),  # FIX: csId 355 → 255
    LabelUnified('curb',                 19, 255, 255,  16,  13,  'barrier',            9,   4,  False, False, (220,190, 40),   25 ),
    LabelUnified('wall',                 20,  12,   3,  17,  14,  'barrier',            9,   4,  False, False, (102,102,156),   3  ),
    LabelUnified('fence',                21,  13,   4,  18,  15,  'barrier',           10,   4,  False, False, (190,153,153),   4  ),
    LabelUnified('guard rail',           22,  14, 255,  19,  16,  'barrier',           10,   4,  False, False, (180,165,180),   26 ),
    LabelUnified('billboard',            23, 255, 255,  20,  17,  'structures',        11,   4,  False, False, (174, 64, 67),   27 ),
    LabelUnified('traffic sign',         24,  20,   7,  21,  18,  'structures',        11,   4,  False, False, (220,220,  0),   7  ),
    LabelUnified('traffic light',        25,  19,   6,  22,  19,  'structures',        11,   4,  False, False, (250,170, 30),   6  ),
    LabelUnified('pole',                 26,  17,   5,  23,  20,  'structures',        12,   4,  False, False, (153,153,153),   5  ),
    LabelUnified('polegroup',            27,  18, 255,  23,  20,  'structures',        12,   4,  False, False, (153,153,153),   35 ),
    LabelUnified('obs-str-bar-fallback', 28, 255, 255,  24,  21,  'structures',        12,   4,  False, False, (169,187,214),   31 ),
    LabelUnified('building',             29,  11,   2,  25,  22,  'construction',      13,   5,  False, False, ( 70, 70, 70),   2  ),
    LabelUnified('bridge',               30,  15, 255,  26,  23,  'construction',      13,   5,  False, False, (150,100,100),   28 ),
    LabelUnified('tunnel',               31,  16, 255,  26,  23,  'construction',      13,   5,  False, False, (150,120, 90),   29 ),
    LabelUnified('vegetation',           32,  21,   8,  27,  24,  'vegetation',        14,   5,  False, False, (107,142, 35),   8  ),
    LabelUnified('sky',                  33,  23,  10,  28,  25,  'sky',               15,   6,  False, False, ( 70,130,180),   10 ),
    LabelUnified('fallback background',  34, 255, 255,  29,  25,  'object fallback',   15,   6,  False, False, (169,187,214),   34 ),
    # ── ignore classes ────────────────────────────────────────────────────────
    LabelUnified('unlabeled',            35,   0, 255, 255, 255,  'void',             255, 255,  False, True,  (  0,  0,  0),   IGNORE_ID),
    LabelUnified('ego vehicle',          36,   1, 255, 255, 255,  'void',             255, 255,  False, True,  (  0,  0,  0),   IGNORE_ID),
    LabelUnified('rectification border', 37,   2, 255, 255, 255,  'void',             255, 255,  False, True,  (  0,  0,  0),   IGNORE_ID),
    LabelUnified('out of roi',           38,   3, 255, 255, 255,  'void',             255, 255,  False, True,  (  0,  0,  0),   IGNORE_ID),
    LabelUnified('license plate',        39, 255, 255, 255, 255,  'vehicle',          255, 255,  False, True,  (  0,  0,142),   IGNORE_ID),
]

# ── Build lookup dicts FROM labels_unified (LabelUnified objects) ─────────────
labels       = labels_unified
name2label   = { l.name : l for l in labels_unified }
id2label     = { l.id   : l for l in labels_unified }
name2label_u = name2label
id2label_u   = id2label

# ── Downstream helpers ────────────────────────────────────────────────────────
IDD_TO_UNIFIED = { l.id: l.unifiedId for l in labels_unified }
UNIFIED_NAMES  = { l.unifiedId: l.name for l in labels_unified if l.unifiedId != IGNORE_ID }
NUM_CLASSES    = 36   # 0–35 valid classes, 255 = ignore

def remap_mask(mask_array, mapping_dict, ignore_id=IGNORE_ID):
    """Remap a 2D numpy label mask using a {src_id: unified_id} dict."""
    out = np.full_like(mask_array, ignore_id, dtype=np.uint8)
    for src_id, unified_id in mapping_dict.items():
        out[mask_array == src_id] = unified_id
    return out

# ── Sanity checks ─────────────────────────────────────────────────────────────
print("✓ Global label lookups now point to unified labels")
print(f"  name2label has {len(name2label)} entries")
print(f"  Sample: car          → unifiedId={name2label['car'].unifiedId}")
print(f"  Sample: autorickshaw → unifiedId={name2label['autorickshaw'].unifiedId}")
print(f"✓ IDD unified labels ready — {NUM_CLASSES} classes (0–35), ignore=255")
print(f"\n{'ID':>4}  {'Unified Name':<25}  {'IDD raw id':>10}")
print("-" * 45)
seen = {}
for l in labels_unified:
    if l.unifiedId == IGNORE_ID:
        continue
    if l.unifiedId not in seen:
        seen[l.unifiedId] = l.name
        marker = "  [CS]" if l.unifiedId <= 18 else "  [IDD extra]"
        print(f"{l.unifiedId:>4}  {l.name:<25}  (idd id={l.id:>2}){marker}")

✓ Global label lookups now point to unified labels
  name2label has 40 entries
  Sample: car          → unifiedId=13
  Sample: autorickshaw → unifiedId=22
✓ IDD unified labels ready — 36 classes (0–35), ignore=255

  ID  Unified Name               IDD raw id
---------------------------------------------
   0  road                       (idd id= 0)  [CS]
  19  parking                    (idd id= 1)  [IDD extra]
  32  drivable fallback          (idd id= 2)  [IDD extra]
   1  sidewalk                   (idd id= 3)  [CS]
  20  rail track                 (idd id= 4)  [IDD extra]
  33  non-drivable fallback      (idd id= 5)  [IDD extra]
  11  person                     (idd id= 6)  [CS]
  21  animal                     (idd id= 7)  [IDD extra]
  12  rider                      (idd id= 8)  [CS]
  17  motorcycle                 (idd id= 9)  [CS]
  18  bicycle                    (idd id=10)  [CS]
  22  autorickshaw               (idd id=11)  [IDD extra]
  13  car                        (idd id=

In [2]:
# ── Annotation data classes ───────────────────────────────────────────────────
Point = namedtuple('Point', ['x', 'y'])

class CsObject:
    def __init__(self):
        self.label    = ""
        self.polygon  = []
        self.id       = -1
        self.deleted  = 0
        self.verified = 0
        self.date     = ""
        self.user     = ""
        self.draw     = True

    def fromJsonText(self, jsonText, objId):
        self.id       = objId
        self.label    = str(jsonText['label'])
        self.polygon  = [Point(p[0], p[1]) for p in jsonText['polygon']]
        self.deleted  = jsonText.get('deleted',  0)
        self.verified = jsonText.get('verified', 1)
        self.user     = jsonText.get('user',    '')
        self.date     = jsonText.get('date',    '')
        self.draw     = (self.deleted != 1)

class Annotation:
    def __init__(self, imageWidth=0, imageHeight=0):
        self.imgWidth  = imageWidth
        self.imgHeight = imageHeight
        self.objects   = []

    def fromJsonFile(self, jsonFile):
        if not os.path.isfile(jsonFile):
            print(f"JSON not found: {jsonFile}")
            return
        with open(jsonFile, 'r') as f:
            d = json.load(f)
        self.imgWidth  = int(d['imgWidth'])
        self.imgHeight = int(d['imgHeight'])
        self.objects   = []
        for i, obj in enumerate(d['objects']):
            o = CsObject()
            o.fromJsonText(obj, i)
            self.objects.append(o)

print("✓ Annotation classes ready")

✓ Annotation classes ready


In [3]:
def _background_val(encoding):
    """Return the background/ignore pixel value for a given encoding."""
    bg_map = {
        'id'        : 255,
        'csId'      : 255,
        'csTrainId' : 255,
        'level4Id'  : 255,
        'level3Id'  : 255,
        'level2Id'  : 255,
        'level1Id'  : 255,
        'unifiedId' : IGNORE_ID,
        'color'     : (0, 0, 0, 0),
    }
    return bg_map.get(encoding, 255)

def _get_label_val(labelTuple, encoding):
    """Return the pixel value for a label given the chosen encoding."""
    return {
        'id'        : labelTuple.id,
        'csId'      : labelTuple.csId,
        'csTrainId' : labelTuple.csTrainId,
        'level4Id'  : labelTuple.level4Id,
        'level3Id'  : labelTuple.level3Id,
        'level2Id'  : labelTuple.level2Id,
        'level1Id'  : labelTuple.level1Id,
        'color'     : labelTuple.color,
        'unifiedId' : labelTuple.unifiedId,
    }.get(encoding, None)

def create_semantic_image(annotation, encoding='level3Id'):
    size = (annotation.imgWidth, annotation.imgHeight)
    bg   = _background_val(encoding)
    mode = "RGBA" if encoding == 'color' else "L"
    img  = Image.new(mode, size, bg)
    draw = ImageDraw.Draw(img)

    for obj in annotation.objects:
        label = obj.label
        if obj.deleted or len(obj.polygon) < 3:
            continue
        if label not in name2label:
            if label.endswith('group'):
                label = label[:-5]
            if label not in name2label:
                tqdm.write(f"  SKIP unknown label: '{label}'")
                continue
        lbl = name2label[label]
        val = _get_label_val(lbl, encoding)
        if val is None:
            tqdm.write(f"  SKIP label '{label}' has no encoding '{encoding}'")
            continue
        draw.polygon(obj.polygon, fill=val)
    return img

def create_instance_image(annotation, encoding='level3Id'):
    size = (annotation.imgWidth, annotation.imgHeight)
    bg   = _background_val(encoding)
    img  = Image.new("I", size, bg)
    draw = ImageDraw.Draw(img)

    # FIX: key counter on the actual encoded val, not level3Id.
    # This prevents pixel collisions when multiple fine-grained labels
    # share the same coarse encoding (e.g. person+rider both → level1Id=2).
    nb_instances = defaultdict(int)

    for obj in annotation.objects:
        label    = obj.label
        is_group = False
        if obj.deleted or len(obj.polygon) < 2:
            continue
        if label not in name2label:
            if label.endswith('group'):
                label, is_group = label[:-5], True
            if label not in name2label:
                tqdm.write(f"  SKIP unknown label: '{label}'")
                continue
        lt  = name2label[label]
        val = _get_label_val(lt, encoding)
        if val is None:
            tqdm.write(f"  SKIP label '{label}' has no encoding '{encoding}'")
            continue
        if lt.hasInstances and not is_group:
            # FIX: was nb_instances[lt.level3Id] — wrong key when encoding
            # is coarser than level3Id (e.g. level1Id, level2Id, unifiedId).
            # Key and increment on val so each encoded class gets its own
            # independent counter and no two instances share a pixel value.
            val = val * 1000 + nb_instances[val]
            nb_instances[val // 1000] += 1
        if val < 0:
            continue
        draw.polygon(obj.polygon, fill=val)
    return img

def create_panoptic_image(instance_array, categories_dict):
    """
    Convert an instance label array into a COCO-format panoptic PNG + segments_info.
    instance_array: 2D numpy array where pixel = classId * 1000 + instanceIndex
    Returns: (RGB PIL Image, list of segment dicts)
    """
    h, w = instance_array.shape
    pan_img = np.zeros((h, w, 3), dtype=np.uint8)
    segments_info = []
    seen_ids = {}

    for y in range(h):
        for x in range(w):
            val = int(instance_array[y, x])
            if val <= 0:
                continue
            class_id    = val // 1000
            instance_id = val  % 1000

            if val not in seen_ids:
                r = (val & 0xFF0000) >> 16
                g = (val & 0x00FF00) >> 8
                b = (val & 0x0000FF)
                seen_ids[val] = (r, g, b)

                cat = categories_dict.get(class_id, {})
                segments_info.append({
                    "id":          val,
                    "category_id": class_id,
                    "iscrowd":     0,
                    "isthing":     cat.get("isthing", 0),
                    "area":        int(np.sum(instance_array == val)),
                })

            pan_img[y, x] = seen_ids[val]

    return Image.fromarray(pan_img, mode="RGB"), segments_info

In [4]:
# ── Per-file worker ───────────────────────────────────────────────────────────
def process_file(args_tuple):
    (json_path, encoding, do_semantic, do_instance,
     do_color, do_panoptic, pan_out_folder,
     categories_dict, out_basedir) = args_tuple

    ann = Annotation()
    ann.fromJsonFile(json_path)

    # ── Windows-safe path handling ────────────────────────────────────────────
    parts     = json_path.replace("\\", "/").split("/")
    try:
        anchor = parts.index("gtFine")
    except ValueError:
        anchor = len(parts) - 4
    rel_parts = parts[anchor:]
    out_dir   = os.path.join(out_basedir, *rel_parts[:-1])
    os.makedirs(out_dir, exist_ok=True)
    stem      = parts[-1].replace("_polygons.json", "")

    # ── semantic ──────────────────────────────────────────────────────────────
    if do_semantic:
        dst = os.path.join(out_dir, f"{stem}_label{encoding}s.png")
        create_semantic_image(ann, encoding).save(dst)

    # ── color visualization ───────────────────────────────────────────────────
    if do_color:
        dst = os.path.join(out_dir, f"{stem}_labelColors.png")
        create_semantic_image(ann, 'color').save(dst)

    # ── instance ──────────────────────────────────────────────────────────────
    inst_img = None
    if do_instance or do_panoptic:
        dst      = os.path.join(out_dir, f"{stem}_instance{encoding}s.png")
        inst_img = create_instance_image(ann, encoding)
        inst_img.save(dst)

    # ── panoptic ──────────────────────────────────────────────────────────────
    if do_panoptic and inst_img is not None:
        inst_arr = np.array(inst_img)
        inst_arr = np.array(
            Image.fromarray(inst_arr).resize((1280, 720), Image.NEAREST)
        )
        pan_img, segm_info = create_panoptic_image(inst_arr, categories_dict)
        city      = parts[anchor + 2] if len(parts) > anchor + 2 else "unknown"
        pan_fname = f"{city}_{stem}_gtFine_panoptic{encoding}s.png"
        pan_img.save(os.path.join(pan_out_folder, pan_fname))
        image_meta = {
            "id":        pan_fname,
            "width":     inst_arr.shape[1],
            "height":    inst_arr.shape[0],
            "file_name": pan_fname,
        }
        return image_meta, segm_info

    return None, None


# ── Master pipeline ───────────────────────────────────────────────────────────
def run_pipeline(
    datadir,
    out_basedir,
    encoding    = 'level3Id',
    do_semantic = True,
    do_instance = False,
    do_color    = False,
    do_panoptic = False,
):
    if do_panoptic:
        do_instance = True

    # ── categories dict ───────────────────────────────────────────────────────
    # FIX: key on _get_label_val(lbl, encoding) instead of hardcoded lbl.level3Id
    # so that class_id recovered from instance pixels (val // 1000) always hits
    # the correct entry regardless of which encoding is active.
    categories, added = [], []
    for lbl in labels:
        if lbl.ignoreInEval:
            continue
        enc_val = _get_label_val(lbl, encoding)
        if enc_val is None or enc_val in added:
            continue
        categories.append({
            'id':            enc_val,
            'name':          lbl.name,
            'color':         lbl.color,
            'supercategory': lbl.level2IdName,
            'isthing':       1 if lbl.hasInstances else 0,
        })
        added.append(enc_val)
    categories_dict = {c['id']: c for c in categories}

    # ── find JSON files ───────────────────────────────────────────────────────
    search = os.path.join(datadir, "gtFine", "*", "*", "*_gt*_polygons.json")
    files  = sorted(glob.glob(search))
    if not files:
        print("No annotation files found. Check DATADIR path and folder structure.")
        print(f"Searched: {search}")
        gtfine = os.path.join(datadir, "gtFine")
        if os.path.isdir(gtfine):
            print(f"Contents of gtFine/: {os.listdir(gtfine)}")
        else:
            print(f"gtFine folder not found at: {gtfine}")
        return

    print(f"Found {len(files)} annotation files")
    print(f"Outputs → {out_basedir}")
    print(f"First file: {files[0]}")

    # ── panoptic output folders ───────────────────────────────────────────────
    pan_folders = {}
    pan_jsons   = {}
    if do_panoptic:
        for split in ['train', 'val']:
            pf = os.path.join(out_basedir, 'gtFine', split + '_panoptic')
            os.makedirs(pf, exist_ok=True)
            pan_folders[split] = pf
            pan_jsons[split]   = {
                'images': [], 'annotations': [], 'categories': categories
            }

    # ── build arg tuples ──────────────────────────────────────────────────────
    arg_tuples = []
    for f in files:
        f_norm  = f.replace("\\", "/")
        split   = 'val' if '/val/' in f_norm else 'train'
        pan_out = pan_folders.get(split, '') if do_panoptic else ''
        arg_tuples.append((f, encoding, do_semantic, do_instance,
                           do_color, do_panoptic, pan_out,
                           categories_dict, out_basedir))

    # ── sequential loop (no multiprocessing — safe on Windows/Jupyter) ────────
    results = []
    for arg in tqdm(arg_tuples, total=len(arg_tuples)):
        try:
            result = process_file(arg)
        except Exception as e:
            tqdm.write(f"  ERROR on {arg[0].split('/')[-1]}: {e}")
            result = (None, None)
        results.append(result)

    # ── write panoptic JSONs ──────────────────────────────────────────────────
    if do_panoptic:
        for f, (image_meta, segm_info) in zip(files, results):
            if image_meta is None:
                continue
            split = 'val' if '/val/' in f.replace("\\", "/") else 'train'
            pan_jsons[split]['images'].append(image_meta)
            pan_jsons[split]['annotations'].append({
                'image_id':      image_meta['id'],
                'file_name':     image_meta['file_name'],
                'segments_info': segm_info,
            })
        for split, data in pan_jsons.items():
            out_path = os.path.join(out_basedir, 'gtFine', f'{split}_panoptic.json')
            with open(out_path, 'w') as jf:
                json.dump(data, jf)
            print(f"Saved panoptic JSON → {out_path}")

    print("✓ Done!")

print("✓ Pipeline ready — sequential mode (Windows/Jupyter safe)")

✓ Pipeline ready — sequential mode (Windows/Jupyter safe)


In [5]:
import os
import json
import glob
from PIL import Image, ImageDraw
from tqdm import tqdm
import numpy as np

In [6]:
DATADIR     = r"E:\Datasets_ALL\Traffic Dataset\IDD-Segmentation"
OUT_BASEDIR = r"E:\Datasets_ALL\Traffic Dataset\IDD-Segmentation\LabelLevel2Id"

run_pipeline(
    datadir     = DATADIR,
    out_basedir = OUT_BASEDIR,

    # ── Label encoding level ───────────────────────────────────────────────
    # 'level3Id'  → 26 classes, recommended for IDD training (default)
    # 'level4Id'  → 29 classes, slightly finer than level3
    # 'id'        → all 40 raw labels including ignored ones (ego vehicle etc.)
    # 'csId'      → Cityscapes-compatible IDs (autorickshaw etc. → 255)
    # 'csTrainId' → 19 Cityscapes benchmark training classes (most coarse)
    # 'level2Id'  → 15 classes, mid-coarse
    # 'level1Id'  → 6 classes, broadest grouping
    # 'unifiedId'  → 36 classes (0–35) remapped from raw IDs, with ignored classes set to 255 (recommended for training/eval on IDD)
    encoding    = 'level2Id',

    # ── Output types (mix and match freely) ───────────────────────────────
    # Grayscale PNG where each pixel value = integer label ID
    # → used directly as training targets for semantic segmentation models
    do_semantic = False,

    # 32-bit PNG where pixel = classId * 1000 + instanceIndex
    # → used for instance segmentation training (Mask R-CNN etc.)
    # → also auto-enabled when do_panoptic = True
    do_instance = True,

    # RGBA PNG with each class painted its defined RGB color
    # → human visualization only, never used for training
    do_color    = False,

    # RGB panoptic PNG (unique color per segment) + COCO-format JSON
    # → used for panoptic segmentation training (Panoptic FPN etc.)
    # → automatically enables do_instance as it is required internally
    do_panoptic = False,
)

Found 16063 annotation files
Outputs → E:\Datasets_ALL\Traffic Dataset\IDD-Segmentation\LabelLevel2Id
First file: E:\Datasets_ALL\Traffic Dataset\IDD-Segmentation\gtFine\train\0\005506_gtFine_polygons.json


  0%|          | 0/16063 [00:00<?, ?it/s]C:\Users\Deus\AppData\Local\Temp\ipykernel_7392\721934790.py:36: DeprecationWarning: Saving I mode images as PNG is deprecated and will be removed in Pillow 13 (2026-10-15)
  inst_img.save(dst)
100%|██████████| 16063/16063 [32:51<00:00,  8.15it/s]

✓ Done!
